# Lab 3: Retrieval Evaluation
## Mean Average Precision, Precision-Recall Curves, and MRR

---

Good retrieval is not enough — we need to *measure* it. This lab introduces the key metrics used to evaluate ranked retrieval systems:

| Metric | What it measures |
|--------|------------------|
| **Precision@k** | Fraction of top-k results that are relevant |
| **Recall@k** | Fraction of all relevant documents found in top-k |
| **Precision-Recall (PR) Curve** | The precision–recall trade-off across all cutoffs |
| **Average Precision (AP)** | Area under the PR curve for a single query |
| **Mean Average Precision (MAP)** | AP averaged over all queries |
| **Mean Reciprocal Rank (MRR)** | How high up is the first relevant result, on average |

### Learning Objectives

By the end of this lab you will be able to:
- Implement Precision@k, Recall@k, AP, MAP, and MRR from scratch
- Plot and interpret Precision-Recall curves
- Compare two retrieval systems using these metrics
- Index a small corpus in OpenSearch and evaluate LMD retrieval against gold relevance judgements

---

### The Evaluation Framework

Every IR evaluation experiment requires three ingredients:

1. **Corpus** — the collection of documents to search over.
2. **Queries** — a set of information needs to evaluate.
3. **Relevance Judgements (qrels)** — for each query, which documents are relevant? These are usually collected from human assessors.

A **retrieval system** returns a **ranked list** of documents for each query. We measure how well those ranked lists agree with the relevance judgements.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import random

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
random.seed(42)
np.random.seed(42)

---

## Part 1 — Understanding Retrieval Metrics

We start with a **toy example** so we can build intuition before touching real data.

### 1.1  Precision and Recall

Given a query, let:
- $R$ = set of **relevant** documents
- $A_k$ = set of documents **retrieved** at rank ≤ $k$

$$\text{Precision@k} = \frac{|R \cap A_k|}{k} \qquad \text{Recall@k} = \frac{|R \cap A_k|}{|R|}$$

- **Precision** answers: *"Of what I returned, how much is useful?"*
- **Recall** answers: *"Of what is useful, how much did I find?"*

They trade off against each other: retrieving more documents increases recall but usually decreases precision.

In [ ]:
# ── Toy Example ──────────────────────────────────────────────────────────────
# A corpus of 10 documents. Documents marked True are relevant to our query.
#                       d1     d2     d3     d4     d5     d6     d7     d8     d9    d10
relevance_labels = [True, False, True, False, True, False, False, True, False, True]

# System A returns documents in this order (document indices, 0-based)
ranking_A = [0, 2, 5, 7, 1, 4, 9, 3, 6, 8]   # a decent system
# System B returns documents in a different order
ranking_B = [1, 5, 0, 3, 9, 6, 2, 7, 4, 8]   # a mediocre system

# Total number of relevant documents
n_relevant = sum(relevance_labels)
print(f"Total relevant documents: {n_relevant} / {len(relevance_labels)}")
print(f"Relevant doc IDs: {[i for i, r in enumerate(relevance_labels) if r]}")

In [ ]:
def precision_at_k(ranking, relevance, k):
    """Fraction of top-k retrieved documents that are relevant."""

    # Get the top-k retrieved document IDs
    top_k = ranking[:k]

    # Count how many of the top-k documents are relevant
    n_relevant_in_top_k = sum(relevance[doc_id] for doc_id in top_k)

    # Precision is the fraction of retrieved documents that are relevant
    return n_relevant_in_top_k / k

def recall_at_k(ranking, relevance, k):
    """Fraction of all relevant documents that appear in top-k."""
    top_k = ranking[:k]

    # Count how many of the top-k documents are relevant
    n_relevant_in_top_k = sum(relevance[doc_id] for doc_id in top_k)

    # Total number of relevant documents in the corpus
    total_relevant = sum(relevance)

    # Recall is the fraction of relevant documents that are retrieved in the top-k
    return n_relevant_in_top_k / total_relevant if total_relevant > 0 else 0.0

# Show P@k and R@k at different cutoffs
print(f"{'k':>4} | {'P@k (A)':>9} | {'R@k (A)':>9} | {'P@k (B)':>9} | {'R@k (B)':>9}")
print("-" * 52)
for k in [1, 2, 3, 5, 7, 10]:
    pa = precision_at_k(ranking_A, relevance_labels, k)
    ra = recall_at_k(ranking_A, relevance_labels, k)
    pb = precision_at_k(ranking_B, relevance_labels, k)
    rb = recall_at_k(ranking_B, relevance_labels, k)
    print(f"{k:>4} | {pa:>9.3f} | {ra:>9.3f} | {pb:>9.3f} | {rb:>9.3f}")

In [ ]:
# ── Same computation using scikit-learn ──────────────────────────────────────
from sklearn.metrics import precision_score, recall_score

def precision_at_k_sklearn(ranking, relevance, k):
    """Precision@k using sklearn.metrics.precision_score."""
    y_true = [int(r) for r in relevance]
    y_pred = [0] * len(relevance)
    for doc_id in ranking[:k]:
        y_pred[doc_id] = 1
    return precision_score(y_true, y_pred, zero_division=0)

def recall_at_k_sklearn(ranking, relevance, k):
    """Recall@k using sklearn.metrics.recall_score."""
    y_true = [int(r) for r in relevance]
    y_pred = [0] * len(relevance)
    for doc_id in ranking[:k]:
        y_pred[doc_id] = 1
    return recall_score(y_true, y_pred, zero_division=0)

print(f"{'k':>4} | {'P@k (A)':>9} | {'R@k (A)':>9} | {'P@k (B)':>9} | {'R@k (B)':>9}")
print("-" * 52)
for k in [1, 2, 3, 5, 7, 10]:
    pa = precision_at_k_sklearn(ranking_A, relevance_labels, k)
    ra = recall_at_k_sklearn(ranking_A, relevance_labels, k)
    pb = precision_at_k_sklearn(ranking_B, relevance_labels, k)
    rb = recall_at_k_sklearn(ranking_B, relevance_labels, k)
    print(f"{k:>4} | {pa:>9.3f} | {ra:>9.3f} | {pb:>9.3f} | {rb:>9.3f}")

**Observations to discuss:**
- At k=10 (full corpus), recall is always 1.0 — we found everything.
- System A has higher precision at low k because it ranks relevant docs earlier.
- Precision generally *decreases* as k grows (we pick up more non-relevant docs).

---

### 1.2  The Precision-Recall Curve

Instead of looking at a single cutoff, we can plot **(Recall, Precision) pairs** at every rank position where a relevant document appears. This gives us the **PR curve**.

A system with a PR curve that is *higher and to the right* is better — it achieves high precision even at high recall.

#### How the Precision-Recall Curve is computed

A ranked retrieval system assigns a **relevance score** $s(d)$ to every document $d$ in the corpus. By sweeping a threshold $\tau$ from high to low, we progressively grow the set of retrieved documents:

$$A_\tau = \{d : s(d) \geq \tau\}$$

At each threshold we record one **operating point**:

$$\text{Precision}(\tau) = \frac{|R \cap A_\tau|}{|A_\tau|} \qquad \text{Recall}(\tau) = \frac{|R \cap A_\tau|}{|R|}$$

where $R$ is the set of relevant documents. Plotting $\bigl(\text{Recall}(\tau),\, \text{Precision}(\tau)\bigr)$ as $\tau$ decreases traces the **PR curve** from top-left (high precision, low recall) towards bottom-right (low precision, high recall).

In a discrete ranked list the threshold only changes meaningfully at each rank position, so the curve has exactly $N$ operating points — one per document. Precision *drops* each time a non-relevant document is retrieved, and *jumps up* each time a relevant document is retrieved, producing the characteristic **staircase** shape.

In [ ]:
def pr_curve(ranking, relevance):
    """
    Compute the Precision-Recall curve for a single query.
    Returns (recall_points, precision_points) at each rank where
    a relevant document is encountered.
    """
    total_relevant = sum(relevance)
    precisions, recalls = [], []
    n_relevant_seen = 0

    for rank, doc_id in enumerate(ranking, start=1):
        if relevance[doc_id]:          # relevant document found at this rank
            n_relevant_seen += 1
            p = n_relevant_seen / rank
            r = n_relevant_seen / total_relevant
            precisions.append(p)
            recalls.append(r)

    return recalls, precisions


def interpolated_pr_curve(recalls, precisions, n_points=11):
    """
    11-point interpolated PR curve (standard in TREC evaluations).
    At each recall level r in {0.0, 0.1, ..., 1.0}, take the
    maximum precision at any recall >= r.
    """
    recall_levels = np.linspace(0, 1, n_points)
    interp_precisions = []
    for r_level in recall_levels:
        # max precision achieved at recall >= r_level
        relevant_precisions = [p for r, p in zip(recalls, precisions) if r >= r_level]
        interp_precisions.append(max(relevant_precisions) if relevant_precisions else 0.0)
    return recall_levels, np.array(interp_precisions)


# Compute PR curves
recalls_A, precisions_A = pr_curve(ranking_A, relevance_labels)
recalls_B, precisions_B = pr_curve(ranking_B, relevance_labels)

ir_A, ip_A = interpolated_pr_curve(recalls_A, precisions_A)
ir_B, ip_B = interpolated_pr_curve(recalls_B, precisions_B)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (r_A, p_A, r_B, p_B, title) in zip(
    axes,
    [
        (recalls_A, precisions_A, recalls_B, precisions_B, "Raw PR Curve"),
        (ir_A, ip_A, ir_B, ip_B, "11-point Interpolated PR Curve"),
    ]
):
    ax.step(r_A, p_A, where='post', color='steelblue', lw=2, label='System A')
    ax.step(r_B, p_B, where='post', color='tomato', lw=2, label='System B')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(title)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.10])
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Reading the PR Curve:**
- The **x-axis** moves right as we retrieve more documents (recall grows).
- The **y-axis** shows the precision at that recall level.
- The *staircase* shape of the raw curve: precision jumps up when a new relevant doc is found, then drops as we continue past non-relevant docs.
- The **interpolated curve** smooths this by taking the *maximum* precision achievable at or above each recall level — this avoids penalizing a system for a slightly unlucky ordering.

---

### 1.3  Average Precision (AP)

The PR curve summarises performance visually. **Average Precision** compresses it to a single number — the **area under the PR curve**.

In practice, AP is computed as the mean of Precision@k *only at ranks where a relevant document appears*:

$$\text{AP} = \frac{1}{|R|} \sum_{k=1}^{N} \text{P@k} \cdot \text{rel}(k)$$

where $\text{rel}(k) = 1$ if the document at rank $k$ is relevant, 0 otherwise.

> A high AP requires relevant documents to appear **early** in the ranking. Placing a relevant doc at rank 1 contributes more than placing it at rank 10.

In [ ]:
def average_precision(ranking, relevance):
    """
    Average Precision for a single query.
    """
    total_relevant = sum(relevance)
    if total_relevant == 0:
        return 0.0

    score = 0.0
    n_relevant_seen = 0
    for rank, doc_id in enumerate(ranking, start=1):
        if relevance[doc_id]:
            n_relevant_seen += 1
            score += n_relevant_seen / rank   # precision at this rank

    return score / total_relevant


ap_A = average_precision(ranking_A, relevance_labels)
ap_B = average_precision(ranking_B, relevance_labels)

print(f"Average Precision — System A: {ap_A:.4f}")
print(f"Average Precision — System B: {ap_B:.4f}")

# ── Visualise where relevant docs land in each ranking ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 3))
for ax, (ranking, name, ap) in zip(axes, [(ranking_A, 'System A', ap_A), (ranking_B, 'System B', ap_B)]):
    colors = ['gold' if relevance_labels[doc_id] else 'lightgrey' for doc_id in ranking]
    ax.bar(range(1, len(ranking)+1), [1]*len(ranking), color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xlabel('Rank')
    ax.set_yticks([])
    ax.set_title(f"{name}  (AP = {ap:.3f})")
    ax.set_xlim([0.4, len(ranking)+0.6])
    relevant_patch = mpatches.Patch(color='gold', label='Relevant')
    non_patch = mpatches.Patch(color='lightgrey', label='Non-relevant')
    ax.legend(handles=[relevant_patch, non_patch], loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

**Intuition check:** Even if both systems eventually retrieve all relevant documents, the system that retrieves them *earlier* gets a higher AP.

---

### 1.4  Mean Average Precision (MAP)

A single query is not enough to reliably evaluate a system. **MAP** averages AP across all queries in the test set:

$$\text{MAP} = \frac{1}{|Q|} \sum_{q \in Q} \text{AP}(q)$$

MAP is the *de-facto* standard metric for ranked retrieval evaluation.

In [ ]:
# Simulate a multi-query evaluation: 6 queries, each with its own
# relevance labels and rankings from two systems.
# We generate them programmatically for reproducibility.

def random_ranking(n_docs, n_relevant, system_quality):
    """
    Generate a (relevance_labels, ranking) pair.
    system_quality in [0,1]: probability that a relevant doc appears in top half.
    """
    doc_ids = list(range(n_docs))
    relevant_ids = set(random.sample(doc_ids, n_relevant))
    labels = [i in relevant_ids for i in range(n_docs)]

    # Build ranking: with probability `system_quality`, place relevant docs first
    rel = [d for d in doc_ids if d in relevant_ids]
    non_rel = [d for d in doc_ids if d not in relevant_ids]
    random.shuffle(rel)
    random.shuffle(non_rel)

    ranking = []
    ri, ni = 0, 0
    for pos in range(n_docs):
        pick_relevant = (ri < len(rel)) and (
            (ni >= len(non_rel)) or (random.random() < system_quality)
        )
        if pick_relevant:
            ranking.append(rel[ri]); ri += 1
        else:
            ranking.append(non_rel[ni]); ni += 1

    return labels, ranking


N_QUERIES = 6
N_DOCS    = 20
N_REL     = 5

random.seed(7)
queries_A, queries_B = [], []
for _ in range(N_QUERIES):
    labels, _ = random_ranking(N_DOCS, N_REL, 0.7)   # same relevance labels
    _, rank_A  = random_ranking(N_DOCS, N_REL, 0.75)  # high-quality system
    _, rank_B  = random_ranking(N_DOCS, N_REL, 0.40)  # lower-quality system
    queries_A.append((labels, rank_A))
    queries_B.append((labels, rank_B))


def mean_average_precision(queries):
    """MAP over a list of (relevance_labels, ranking) pairs."""
    return np.mean([average_precision(rank, labels) for labels, rank in queries])


map_A = mean_average_precision(queries_A)
map_B = mean_average_precision(queries_B)

# Per-query AP breakdown
print(f"{'Query':>7} | {'AP (A)':>8} | {'AP (B)':>8}")
print("-" * 30)
for i, (qa, qb) in enumerate(zip(queries_A, queries_B), start=1):
    print(f"{'Q'+str(i):>7} | {average_precision(qa[1], qa[0]):>8.4f} | {average_precision(qb[1], qb[0]):>8.4f}")
print("-" * 30)
print(f"{'MAP':>7} | {map_A:>8.4f} | {map_B:>8.4f}")

---

### 1.5  Mean Reciprocal Rank (MRR)

Sometimes you only care about whether the **first** relevant result is near the top — for example, in question answering or navigational search, users expect the answer in position 1 or 2.

**Reciprocal Rank (RR)** for a single query:

$$\text{RR} = \frac{1}{\text{rank of first relevant document}}$$

If the first relevant doc is at rank 1 → RR = 1.0  
If it's at rank 2 → RR = 0.5  
If it's at rank 5 → RR = 0.2  

**Mean Reciprocal Rank** averages over queries:

$$\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \text{RR}(q)$$

> MRR is sensitive *only* to the position of the first hit. MAP rewards systems that retrieve *all* relevant documents early.

In [ ]:
def reciprocal_rank(ranking, relevance):
    """1 / (rank of the first relevant document). 0 if none found."""
    for rank, doc_id in enumerate(ranking, start=1):
        if relevance[doc_id]:
            return 1.0 / rank
    return 0.0


def mean_reciprocal_rank(queries):
    """MRR over a list of (relevance_labels, ranking) pairs."""
    return np.mean([reciprocal_rank(rank, labels) for labels, rank in queries])


mrr_A = mean_reciprocal_rank(queries_A)
mrr_B = mean_reciprocal_rank(queries_B)

print(f"{'Query':>7} | {'RR (A)':>8} | {'RR (B)':>8}")
print("-" * 30)
for i, (qa, qb) in enumerate(zip(queries_A, queries_B), start=1):
    print(f"{'Q'+str(i):>7} | {reciprocal_rank(qa[1], qa[0]):>8.4f} | {reciprocal_rank(qb[1], qb[0]):>8.4f}")
print("-" * 30)
print(f"{'MRR':>7} | {mrr_A:>8.4f} | {mrr_B:>8.4f}")
print(f"{'MAP':>7} | {map_A:>8.4f} | {map_B:>8.4f}")

---

### 1.6  Visualising System Comparison

Let's aggregate the PR curves across all queries (by interpolation) and plot them side by side with the metric scores.

In [ ]:
def mean_pr_curve(queries, n_points=11):
    """Average interpolated PR curve over multiple queries."""
    all_interp = []
    for labels, ranking in queries:
        r, p = pr_curve(ranking, labels)
        _, ip = interpolated_pr_curve(r, p, n_points)
        all_interp.append(ip)
    recall_levels = np.linspace(0, 1, n_points)
    return recall_levels, np.mean(all_interp, axis=0)


rl_A, mean_p_A = mean_pr_curve(queries_A)
rl_B, mean_p_B = mean_pr_curve(queries_B)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Mean PR curves ───────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(rl_A, mean_p_A, 'o-', color='steelblue', lw=2, label=f'System A  (MAP={map_A:.3f})')
ax.plot(rl_B, mean_p_B, 's-', color='tomato',    lw=2, label=f'System B  (MAP={map_B:.3f})')
ax.fill_between(rl_A, mean_p_A, alpha=0.1, color='steelblue')
ax.fill_between(rl_B, mean_p_B, alpha=0.1, color='tomato')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Mean Interpolated PR Curve (across queries)')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.10])
ax.legend()
ax.grid(alpha=0.3)

# ── Bar chart comparison ──────────────────────────────────────────────────────
ax = axes[1]
metrics = ['MAP', 'MRR']
vals_A  = [map_A, mrr_A]
vals_B  = [map_B, mrr_B]
x = np.arange(len(metrics))
width = 0.35
bars_A = ax.bar(x - width/2, vals_A, width, label='System A', color='steelblue', alpha=0.85)
bars_B = ax.bar(x + width/2, vals_B, width, label='System B', color='tomato',    alpha=0.85)
for bar in bars_A + bars_B:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim([0, 1.0])
ax.set_title('Metric Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## Exercise 1 — Metrics by Hand

Below is a query with 3 relevant documents out of 8, and two system rankings.

```
Relevant documents: {A, C, F}

System X ranking:  [A, B, C, D, E, F, G, H]
System Y ranking:  [B, D, A, G, F, H, C, E]
```

**Tasks:**

1. Compute P@3, R@3, P@5, R@5 for both systems.
2. Compute Average Precision (AP) for both systems. Which system is better?
3. Compute Reciprocal Rank (RR) for both systems.
4. Sketch (or plot) the PR curve for both systems on the same axes.
5. *(Bonus)* Add a third system, System Z, where all three relevant documents appear at ranks 6, 7, 8. Compute its AP and RR. What does this tell you about systems that have high recall but low precision at low ranks?

In [ ]:
# ── Exercise 1 – your code here ───────────────────────────────────────────────
# Documents: A=0, B=1, C=2, D=3, E=4, F=5, G=6, H=7
ex_relevance = [True, False, True, False, False, True, False, False]  # A, C, F

ex_ranking_X = [0, 1, 2, 3, 4, 5, 6, 7]  # [A, B, C, D, E, F, G, H]
ex_ranking_Y = [1, 3, 0, 6, 5, 7, 2, 4]  # [B, D, A, G, F, H, C, E]

# TODO: compute the metrics and plot the PR curve


---

## Part 2 — Evaluating Real Retrieval with OpenSearch

Now we apply the same metrics to a real scenario:

1. We have a **small curated corpus** of 30 passages on four topics (NLP, Information Retrieval, Computer Vision, Reinforcement Learning).
2. We have **5 queries** with manually assigned **relevance judgements** (qrels).
3. We will **index the corpus in OpenSearch** and run **LMD (Language Model with Dirichlet smoothing) retrieval**.
4. We will **evaluate** the LMD rankings using MAP, MRR, and PR curves.

### 2.1  Corpus and Relevance Judgements

In [ ]:
# ── Corpus ────────────────────────────────────────────────────────────────────
# 30 short passages across 4 topics.
# Each entry: {"id": str, "topic": str, "text": str}

corpus = [
    # NLP (D01-D08)
    {"id": "D01", "topic": "nlp",
     "text": "BERT is a transformer-based language model pre-trained on masked language modelling and next-sentence prediction, producing deep bidirectional representations."},
    {"id": "D02", "topic": "nlp",
     "text": "GPT models are autoregressive language models trained to predict the next token, enabling powerful open-ended text generation."},
    {"id": "D03", "topic": "nlp",
     "text": "Fine-tuning adapts a large pre-trained language model to a downstream task by continuing training on a small labelled dataset."},
    {"id": "D04", "topic": "nlp",
     "text": "The self-attention mechanism allows a transformer to dynamically weight the importance of every other token when encoding each position."},
    {"id": "D05", "topic": "nlp",
     "text": "Word embeddings such as Word2Vec and GloVe represent words as dense vectors that capture semantic and syntactic relationships."},
    {"id": "D06", "topic": "nlp",
     "text": "Subword tokenisation splits text into pieces like byte-pair encodings, enabling models to handle rare and unseen vocabulary efficiently."},
    {"id": "D07", "topic": "nlp",
     "text": "Named entity recognition (NER) identifies and classifies text spans referring to real-world entities such as people, organisations, and locations."},
    {"id": "D08", "topic": "nlp",
     "text": "Machine translation encodes a source-language sentence and decodes it into a target language using encoder-decoder architectures."},

    # Information Retrieval (D09-D16)
    {"id": "D09", "topic": "ir",
     "text": "BM25 is a probabilistic retrieval function that scores documents by query-term frequency and inverse document frequency, with length normalisation."},
    {"id": "D10", "topic": "ir",
     "text": "Dense retrieval encodes queries and documents into continuous vector spaces and retrieves candidates via approximate nearest-neighbour search."},
    {"id": "D11", "topic": "ir",
     "text": "An inverted index maps each vocabulary term to the list of documents in which it appears, enabling fast keyword-based lookup."},
    {"id": "D12", "topic": "ir",
     "text": "Relevance judgements (qrels) record which documents are relevant to each query and form the gold standard for IR evaluation."},
    {"id": "D13", "topic": "ir",
     "text": "Hybrid retrieval fuses sparse BM25 scores with dense vector scores to leverage complementary lexical and semantic signals."},
    {"id": "D14", "topic": "ir",
     "text": "Learning to rank trains supervised models to optimise ranking metrics such as NDCG directly using human-labelled query-document pairs."},
    {"id": "D15", "topic": "ir",
     "text": "Cross-encoder rerankers jointly encode the query and a candidate document to produce a fine-grained relevance score, at higher computational cost."},
    {"id": "D16", "topic": "ir",
     "text": "Query expansion adds synonyms or related terms to the original query to improve recall in keyword-based search systems."},

    # Computer Vision (D17-D23)
    {"id": "D17", "topic": "cv",
     "text": "Convolutional neural networks learn hierarchical visual features by applying learned filter banks across the spatial dimensions of an image."},
    {"id": "D18", "topic": "cv",
     "text": "Object detection models predict class labels and bounding boxes for every object present in an image."},
    {"id": "D19", "topic": "cv",
     "text": "Semantic segmentation assigns a class label to every pixel in an image, enabling detailed scene understanding."},
    {"id": "D20", "topic": "cv",
     "text": "Vision transformers split images into fixed-size patches and process them with self-attention, achieving competitive results on image classification."},
    {"id": "D21", "topic": "cv",
     "text": "Transfer learning from ImageNet pre-trained weights dramatically reduces the data and compute needed for downstream vision tasks."},
    {"id": "D22", "topic": "cv",
     "text": "Data augmentation techniques—such as random cropping, flipping, and colour jitter—improve model generalisation in image recognition."},
    {"id": "D23", "topic": "cv",
     "text": "Generative adversarial networks train a generator and discriminator in opposition, producing highly realistic synthetic images."},

    # Reinforcement Learning (D24-D30)
    {"id": "D24", "topic": "rl",
     "text": "Reinforcement learning trains an agent to select actions that maximise cumulative discounted reward through trial-and-error interaction with an environment."},
    {"id": "D25", "topic": "rl",
     "text": "Q-learning iteratively estimates the value of taking each action in each state, converging to an optimal policy without a model of the environment."},
    {"id": "D26", "topic": "rl",
     "text": "Policy gradient methods directly optimise the agent's stochastic policy by computing the gradient of the expected cumulative reward."},
    {"id": "D27", "topic": "rl",
     "text": "The exploration-exploitation trade-off requires agents to balance acquiring new knowledge about the environment with exploiting known high-reward strategies."},
    {"id": "D28", "topic": "rl",
     "text": "Deep Q-networks combine Q-learning with deep neural networks to handle high-dimensional state spaces such as raw video frames from Atari games."},
    {"id": "D29", "topic": "rl",
     "text": "Actor-critic methods jointly train a policy network (actor) and a value function network (critic), reducing variance in gradient estimates."},
    {"id": "D30", "topic": "rl",
     "text": "Proximal Policy Optimisation (PPO) stabilises policy gradient training by clipping the objective to limit the size of each policy update."},
]

print(f"Corpus size: {len(corpus)} documents")
for doc in corpus[:3]:
    print(f"  [{doc['id']}] ({doc['topic']}) {doc['text'][:80]}...")

In [ ]:
# ── Queries and Relevance Judgements (qrels) ──────────────────────────────────
# qrels[query_id] = set of relevant document IDs

queries = {
    "Q1": "pre-trained transformers and language model fine-tuning",
    "Q2": "document ranking and search engine retrieval models",
    "Q3": "deep learning for image recognition and visual features",
    "Q4": "reinforcement learning agents maximising rewards",
    "Q5": "how words are encoded as vectors for neural models",
}

qrels = {
    "Q1": {"D01", "D02", "D03", "D04"},          # BERT, GPT, fine-tuning, attention
    "Q2": {"D09", "D10", "D11", "D13", "D14", "D15", "D16"},  # LMD, dense, index, hybrid, L2R, reranker, expansion
    "Q3": {"D17", "D18", "D19", "D20", "D21"},    # CNN, detection, segmentation, ViT, transfer
    "Q4": {"D24", "D25", "D26", "D27", "D28", "D29", "D30"},  # RL, Q-learning, policy gradient, exploration, DQN, A2C, PPO
    "Q5": {"D05", "D06"},                          # embeddings, tokenisation
}

# Sanity check
for qid, relevant in qrels.items():
    print(f"  {qid}: '{queries[qid][:55]}...' → {len(relevant)} relevant docs")

### 2.2  Index the Corpus in OpenSearch

In [ ]:
from opensearchpy import OpenSearch
from opensearchpy.helpers import bulk

HOST     = "api.novasearch.org"
PORT     = 443
USERNAME = ""   # ← change to your username
PASSWORD = ""     # ← change to your password


# Create the client with SSL/TLS enabled, but hostname verification disabled.
client = OpenSearch(
    hosts = [{'host': HOST, 'port': PORT}],
    http_compress = True, # enables gzip compression for request bodies
    http_auth = (USERNAME, PASSWORD),
    use_ssl = True,
    url_prefix = 'opensearch_v3',
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False,
    timeout=30,
)
	

In [ ]:
INDEX_NAME = f"{USERNAME}_lab3_corpus"

# Delete if it already exists
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    print(f"Deleted existing index: {INDEX_NAME}")

# Create index with LMDirichlet similarity (mu=2000 is the standard default)
index_body = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "similarity": {
            "default": {"type": "LMDirichlet", "mu": 2000}
        }
    },
    "mappings": {
        "properties": {
            "doc_id":  {"type": "keyword"},
            "topic":   {"type": "keyword"},
            "text":    {"type": "text", "analyzer": "english"},
        }
    }
}

client.indices.create(index=INDEX_NAME, body=index_body)
print(f"Created index: {INDEX_NAME}")

In [ ]:
# Bulk-index all 30 documents
actions = [
    {
        "_index": INDEX_NAME,
        "_id":    doc["id"],
        "_source": {
            "doc_id": doc["id"],
            "topic":  doc["topic"],
            "text":   doc["text"],
        }
    }
    for doc in corpus
]

success, errors = bulk(client, actions)
client.indices.refresh(index=INDEX_NAME)
print(f"Indexed {success} documents. Errors: {errors}")

### 2.3  LMD Retrieval

In [ ]:
def lmd_search(query_text, index, n_results=30):
    """
    Run an LMD (Language Model with Dirichlet smoothing) query and return a ranked list of doc IDs.
    Scoring is controlled by the LMDirichlet similarity set on the index (mu=2000).
    """
    response = client.search(
        index=index,
        body={
            "size": n_results,
            "query": {
                "match": {
                    "text": {"query": query_text}
                }
            },
            "_source": ["doc_id", "topic"],
        }
    )
    hits = response["hits"]["hits"]
    return [(h["_source"]["doc_id"], h["_score"]) for h in hits]


# Retrieve for each query
lmd_results = {}
for qid, query_text in queries.items():
    results = lmd_search(query_text, INDEX_NAME)
    lmd_results[qid] = results
    ranked_ids = [r[0] for r in results]
    print(f"\n{qid}: {query_text}")
    print(f"  Top-5: {ranked_ids[:5]}")
    print(f"  Relevant: {qrels[qid]}")

### 2.4  Compute Retrieval Metrics

We reuse the metric functions from Part 1. We just need to convert the OpenSearch result lists into the same format.

In [ ]:
def results_to_ranking(results, qrels_set, all_doc_ids):
    """
    Convert OpenSearch results to the (relevance_labels, ranking) format
    expected by our metric functions.

    - results: list of (doc_id, score) from OpenSearch, ranked by score
    - qrels_set: set of relevant doc IDs for this query
    - all_doc_ids: ordered list of all doc IDs (defines index positions)
    """
    id_to_idx = {doc_id: i for i, doc_id in enumerate(all_doc_ids)}

    # relevance_labels: parallel to all_doc_ids
    relevance = [doc_id in qrels_set for doc_id in all_doc_ids]

    # ranking: indices into all_doc_ids, ordered by retrieval rank
    retrieved_ids = [r[0] for r in results]
    # append any doc not retrieved at the end (unranked)
    retrieved_set = set(retrieved_ids)
    not_retrieved = [doc_id for doc_id in all_doc_ids if doc_id not in retrieved_set]
    full_ranking_ids = retrieved_ids + not_retrieved
    ranking = [id_to_idx[doc_id] for doc_id in full_ranking_ids]

    return relevance, ranking


all_doc_ids = [doc["id"] for doc in corpus]

# Build (labels, ranking) pairs for each query
eval_queries = []
for qid in queries:
    labels, ranking = results_to_ranking(lmd_results[qid], qrels[qid], all_doc_ids)
    eval_queries.append((qid, labels, ranking))

# ── Per-query metrics ─────────────────────────────────────────────────────────
print(f"{'Query':>5} | {'AP':>6} | {'RR':>6} | {'P@5':>6} | {'R@5':>6} | {'P@10':>6} | {'R@10':>6}")
print("-" * 58)
all_ap, all_rr = [], []
for qid, labels, ranking in eval_queries:
    ap  = average_precision(ranking, labels)
    rr  = reciprocal_rank(ranking, labels)
    p5  = precision_at_k(ranking, labels, 5)
    r5  = recall_at_k(ranking, labels, 5)
    p10 = precision_at_k(ranking, labels, 10)
    r10 = recall_at_k(ranking, labels, 10)
    all_ap.append(ap); all_rr.append(rr)
    print(f"{qid:>5} | {ap:>6.3f} | {rr:>6.3f} | {p5:>6.3f} | {r5:>6.3f} | {p10:>6.3f} | {r10:>6.3f}")
print("-" * 58)
print(f"{'Mean':>5} | {np.mean(all_ap):>6.3f} | {np.mean(all_rr):>6.3f} |")
print(f"\nMAP  = {np.mean(all_ap):.4f}")
print(f"MRR  = {np.mean(all_rr):.4f}")

### 2.5  Visualise LMD Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Per-query PR curves ───────────────────────────────────────────────────────
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 0.5, len(eval_queries)))
interp_curves = []
for (qid, labels, ranking), color in zip(eval_queries, colors):
    r, p = pr_curve(ranking, labels)
    ir, ip = interpolated_pr_curve(r, p)
    interp_curves.append(ip)
    ax.plot(ir, ip, 'o--', color=color, lw=1.5, ms=4,
            label=f"{qid} (AP={average_precision(ranking, labels):.2f})")

# Mean PR curve
rl = np.linspace(0, 1, 11)
mean_p = np.mean(interp_curves, axis=0)
ax.plot(rl, mean_p, 'k-', lw=3, label=f"Mean (MAP={np.mean(all_ap):.2f})", zorder=5)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('LMD — Interpolated PR Curves per Query')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.10])
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Ranked result heatmap ─────────────────────────────────────────────────────
ax = axes[1]
cutoff = 15   # show only top 15 positions
heat_data = np.zeros((len(eval_queries), cutoff))
for qi, (qid, labels, ranking) in enumerate(eval_queries):
    for rank_pos in range(cutoff):
        heat_data[qi, rank_pos] = 1.0 if labels[ranking[rank_pos]] else 0.0

im = ax.imshow(heat_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(cutoff))
ax.set_xticklabels([str(i+1) for i in range(cutoff)], fontsize=8)
ax.set_yticks(range(len(eval_queries)))
ax.set_yticklabels([q[0] for q in eval_queries])
ax.set_xlabel('Rank position')
ax.set_title('LMD Relevance Heatmap (green = relevant)')
plt.colorbar(im, ax=ax, fraction=0.04)

plt.tight_layout()
plt.show()

**Analysing the results:**
- The heatmap shows where relevant documents land in the LMD ranking for each query.
- A solid green band on the left means LMD retrieved relevant documents early (high AP).
- Gaps indicate non-relevant documents ranked above relevant ones.
- Look at Q5 (`word representations`) — LMD may struggle because the query uses words that do not appear literally in the relevant documents (vocabulary mismatch).

**Questions — analyse the plots above and answer the following:**

1. **Which query has the highest AP and which has the lowest?** Look at the PR curves and the heatmap. What do the top-15 retrieved documents for each query tell you about why LMD succeeds or fails?

2. **Heatmap gaps.** Identify a query where a large block of non-relevant documents (red) appears before the first relevant result. At what rank does the first relevant document appear for that query? How does that affect its Reciprocal Rank (RR)?

3. **High recall, low precision.** Is there a query where the PR curve reaches recall = 1.0 but only at a very low precision level? What does that mean in practice for a user browsing the results?

4. **Vocabulary mismatch.** Q5 asks about *"how words are encoded as vectors for neural models"*. Look at the relevant documents (D05, D06) and compare their vocabulary to the query terms. Can you identify specific words that cause LMD to rank non-relevant documents above D05/D06?

5. **Comparing queries with different numbers of relevant documents.** Q5 has 2 relevant documents while Q4 has 7. How does the number of relevant documents affect the shape of the PR curve and the maximum achievable recall at low cutoffs (e.g., P@5)?

6. **What would it take to improve LMD on the worst-performing query?** Propose one concrete change — for example to the query, the index settings, or the retrieval approach — that you think would raise its AP. Justify your answer."


---

## Exercise 2 — Evaluate and Compare Retrieval Systems

### 2a. Inspect LMD Failures

1. For the query with the **lowest AP**, print the top-10 LMD results and the list of relevant documents. Can you explain why LMD failed? Is it a vocabulary mismatch problem?
2. Compute **P@1**, **P@3**, **P@5**, and **P@10** for every query and display the results in a single table.

### 2b. Implement a Baseline Oracle

An **oracle ranker** knows the relevance judgements and always returns relevant documents first.

1. Implement a function `oracle_ranking(qrels_set, all_doc_ids)` that produces the perfect ranking for a query.
2. Compute MAP and MRR for the oracle. This is the **upper bound** any system can achieve.
3. Plot the oracle's mean PR curve alongside LMD's. What does the gap look like?

### 2c. Dense Retrieval (Optional — requires Lab 1 setup)

1. Use the `sentence-transformers/msmarco-distilbert-base-v2` model from Lab 1 to encode all documents into dense vectors and store them in an additional KNN index.
2. Run dense retrieval for all 5 queries.
3. Compute MAP and MRR for dense retrieval and add it to your comparison table.
4. Does dense retrieval help for Q5 (`word representations`)? Discuss why or why not.

In [ ]:
# ── Exercise 2a — your code here ──────────────────────────────────────────────


In [ ]:
# ── Exercise 2b — Oracle ranker ───────────────────────────────────────────────

def oracle_ranking(qrels_set, all_doc_ids):
    """
    TODO: return (relevance_labels, ranking) where relevant documents
    are placed first in the ranking.
    """
    pass


In [ ]:
# ── Exercise 2c — Dense retrieval (optional) ──────────────────────────────────
